<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_80_multi_tenant_ai_platform_patterns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🏢 **Module 80 — Multi-Tenant AI Platform Patterns** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 10 · Master-Plan Gaps


# Module 80 — Multi-Tenant AI Platform Patterns

> Every AI platform that grows past one product hits the same wall: **how do you let many teams (or paying customers) share one platform without their data, models, costs, or noise leaking into each other?** Get this wrong and a single tenant can leak another's RAG corpus, eat all the GPU budget, or DoS the cluster.
>
> This module is the production-grade map: **isolation models, request routing, per-tenant LoRAs, quotas, cost attribution (M79), data residency, and tenant lifecycle.**

### What you'll cover
1. The four tenant archetypes
2. The four isolation models — shared / namespaced / silo / dedicated
3. **Request path** — how a request finds its tenant
4. **Data isolation** — RAG, embeddings, vector DBs, feature stores
5. **Per-tenant LoRA** — one base model, many fine-tunes
6. **Quotas + rate limits + fair-share scheduling**
7. **Tenant-aware observability** — OTel attributes, cost slicing
8. **Auth + OIDC + scoped tokens**
9. **Tenant lifecycle** — provisioning, suspension, deletion (GDPR / DSAR)
10. The reference architecture + anti-patterns


## 1 · The four tenant archetypes

| Archetype | Tenants are… | Examples | Drives… |
|---|---|---|---|
| **B2B SaaS** | paying customer companies | Notion AI, Intercom Fin, Vanta AI | per-customer data isolation + billing |
| **Internal platform** | other teams in your company | Spotify ML Platform, Uber Michelangelo, Netflix Metaflow | quota / chargeback / showback (M79) |
| **Model marketplace** | end-developers calling your hosted models | OpenAI, Anthropic, Together AI, Fireworks | abuse + per-API-key budgets |
| **Government / regulated** | agencies / banks / hospitals | Palantir, Anthropic Gov, Mistral EU | data residency, audit, FedRAMP / SOC2 / HIPAA |

The architecture you build differs **a lot** by archetype. We'll show the patterns that apply to most; we'll flag the regulated-only ones explicitly.

## 2 · The four isolation models

A spectrum from cheapest+leakiest to most-isolated+most-expensive:

```
   ┌─────────────────────────────────────────────────────────────────────────────┐
   │                                                                              │
   │   shared-everything  →  namespaced  →  silo  →  dedicated stack             │
   │                                                                              │
   │     |←── cheaper                                  more isolated ──→|         │
   │                                                                              │
   └─────────────────────────────────────────────────────────────────────────────┘
```

| Layer | Shared | Namespaced | Silo | Dedicated |
|---|---|---|---|---|
| **DB** | one table, `tenant_id` column | one schema per tenant | one DB per tenant | one DB instance per tenant |
| **Vector store** | one collection, `tenant_id` filter | one collection per tenant | one vector DB per tenant | dedicated cluster |
| **Inference** | one vLLM pool | one pool per tier (silver / gold) | per-tenant LoRA hot-swapped | dedicated vLLM cluster |
| **K8s** | shared cluster, shared namespace | one namespace per tenant | dedicated node pool | dedicated cluster |
| **Cost** | ~$/tenant low | medium | high | very high |
| **Risk** | row-level leak ends you | strong | very strong | gold standard |
| **Tenant size** | startups / SMB | most SaaS | enterprise customers | regulated gov / fintech |

> 🟡 **Most production B2B SaaS platforms run *namespaced* by default and *silo* the top 5-10 % of customers** who pay for it. Pure shared-everything stops scaling at ~thousands of tenants when row-level filters become a leak audit risk.

### A concrete pattern (very common in 2026)

```
   ┌─────── shared k8s ─────────┐
   │  ┌── namespace: tenant-A ──┐  │  vLLM pod   ←─ shared base model
   │  │  app pods, Redis, …    │  │  tenant-A LoRA loaded per request
   │  └────────────────────────┘  │
   │  ┌── namespace: tenant-B ──┐  │  vLLM pod   ←─ shared base model
   │  │  app pods, Redis, …    │  │  tenant-B LoRA loaded per request
   │  └────────────────────────┘  │
   └──────────────────────────────┘
              │
              ▼
   ┌── dedicated stack: tenant-Z (regulated) ──┐
   │  separate cluster + IDP + DB + KMS         │
   └────────────────────────────────────────────┘
```

Namespaced for the long tail, dedicated for the gold customers, one base model serving everyone.

## 3 · The request path

How does a request know which tenant it belongs to? Three steps:

```
   1. Edge / gateway extracts tenant_id from
        - API key (most common)         tenant_id = key_table.lookup(api_key).tenant_id
        - OIDC JWT claim                tenant_id = jwt.claims["tenant_id"]
        - Subdomain                     tenant_id = host.split(".")[0]
        - Header                        tenant_id = headers["X-Tenant-Id"]

   2. The context propagates through every internal hop via:
        - gRPC metadata (M45) OR
        - HTTP header preserved by every service OR
        - explicit arg on tool / MCP / A2A calls (M64, M75)

   3. EVERY downstream — DB, vector store, model router, telemetry — must
      enforce `tenant_id` rather than trust the caller's claim.
```

**The golden rule.** The tenant context is **carried** by the request, but **enforced** at every persistence and resource-allocation point. Never let the caller pass `tenant_id` as a function argument that a downstream service "trusts" — always re-derive from the request principal or a short-lived signed claim.

In [ ]:
# A minimal middleware that does extraction + propagation correctly
from fastapi import FastAPI, Request, HTTPException, Depends
import contextvars

# Async-safe context var
_tenant: contextvars.ContextVar[str | None] = contextvars.ContextVar("tenant", default=None)

def tenant_id() -> str:
    """Read inside any handler/service to know who the request belongs to."""
    t = _tenant.get()
    if not t:
        raise HTTPException(status_code=400, detail="No tenant context")
    return t

app = FastAPI()

@app.middleware("http")
async def tenant_middleware(request: Request, call_next):
    # 1. extract (in priority order)
    tid = request.headers.get("x-tenant-id")
    if not tid and "authorization" in request.headers:
        # in real life: validate JWT, read claim
        tid = "demo-tenant-from-jwt"
    if not tid:
        return await call_next(request)        # let unauthenticated routes pass

    # 2. propagate via contextvar (carries across `await`)
    token = _tenant.set(tid)
    try:
        response = await call_next(request)
    finally:
        _tenant.reset(token)
    return response

@app.get("/api/chat")
async def chat(t: str = Depends(tenant_id)):
    return {"answer": f"hi, tenant {t}!"}

**Two ways propagation breaks.**
1. **Mixed sync/async stacks** — `contextvars` are async-safe but break across thread-pool executors. Always propagate explicitly through gRPC metadata (M45) or A2A `caller` (M75).
2. **Cross-tenant background jobs** — celery / RQ / k8s `Job` workers process tasks from a queue. The tenant must be **on the message**, not the worker's environment.

## 4 · Data isolation — RAG, embeddings, vector DBs, feature stores

The #1 way multi-tenant AI breaks: **tenant A's RAG retrieves tenant B's documents.** Avoid by enforcing tenant scope at *every* read.

### Pattern A — one collection, `tenant_id` filter (cheap)
```python
results = qdrant.search(
    collection_name="docs",
    query_vector=query_emb,
    query_filter=Filter(must=[FieldCondition(key="tenant_id", match=MatchValue(value=tid))]),
    limit=5,
)
```
Works at small scale. Vector DBs that support **filtered ANN** (Qdrant, Pinecone, Weaviate, Vespa, pgvector — M42) make this efficient. Risk: **one typo** in a filter and you serve cross-tenant.

### Pattern B — one collection per tenant
```python
qdrant.search(collection_name=f"docs__{tid}", query_vector=query_emb, limit=5)
```
Stronger isolation; harder to manage at thousands of tenants. Pinecone's "namespaces" and Weaviate's "tenants" support this natively without an extra cluster.

### Pattern C — silo / dedicated vector DB
For your top-paying or regulated customers. Same with Postgres (per-tenant schema or per-tenant DB), the feature store (M69), etc.

> ✅ **The rule.** **Tenant filter on every query**, code-reviewed; integration tests that assert no leakage; audit logs of every retriever call with `tenant_id` attached.

## 5 · Per-tenant LoRA — one base model, many fine-tunes

The killer pattern for multi-tenant LLM apps. Frontier base model is shared (a single vLLM pool with PagedAttention — M44); **each tenant has a small (~20-200 MB) LoRA** (M39) that adapts it to their domain / tone / data.

In [ ]:
# vLLM multi-LoRA: many adapters loaded on one base model, swapped per request
vllm_multi_lora = '''
# Server side — load the base model with --enable-lora and per-tenant adapters
python -m vllm.entrypoints.openai.api_server \
    --model meta-llama/Llama-3.3-70B-Instruct \
    --enable-lora \
    --lora-modules \
        tenant-acme=s3://my-bucket/loras/acme-v2.tar \
        tenant-foo=s3://my-bucket/loras/foo-v1.tar \
        tenant-bar=s3://my-bucket/loras/bar-v3.tar \
    --max-loras 8 \
    --max-lora-rank 64

# Client side — pick the adapter via `model` field
curl http://vllm:8000/v1/chat/completions \
    -H "Authorization: Bearer $API_KEY" \
    -d '{
        "model": "tenant-acme",                    # ← LoRA name
        "messages": [{"role":"user","content":"…"}]
    }'
'''
print(vllm_multi_lora)

**Why this scales.**
- One **70B base** in VRAM, hot, shared across all tenants → KV cache amortised across the fleet (M60).
- **Hundreds of LoRAs** hot-swapped on demand at ~10 ms switch cost.
- Cold tenants paged out to disk; vLLM keeps recently used ones resident.
- vLLM, **SGLang**, **LoRAX (Predibase)**, and Triton (M71) all support this pattern.

**Per-tenant fine-tuning lifecycle:**
1. Tenant ingests training data (chat logs, docs) via your platform UI.
2. Backend kicks a **QLoRA** (M39) job on a small GPU pool.
3. The new adapter is signed, uploaded to S3, and registered in the model catalog.
4. The router updates the LoRA registry. Next request from tenant-acme uses `tenant-acme-v2`.
5. Old adapters retained for **30 days** for rollback; then GC'd.

> 🔒 **Security note.** A poisoned LoRA can break the base model's safety training. **Scan adapter datasets** (M74) before training, sign the adapter on completion, verify the signature at load.

## 6 · Quotas + rate limits + fair-share scheduling

A single greedy tenant will eat your platform if you let it. Four layers of defence:

| Layer | Tool | Enforces |
|---|---|---|
| **Edge gateway** | API gateway (Kong, Envoy, Apigee), or LiteLLM, or custom | RPS / TPS / monthly $ |
| **Token-level meter** | LiteLLM, Helicone — counts tokens per call | $/Mtok per tenant tier |
| **k8s ResourceQuota / LimitRange** | namespace quotas (M46) | CPU / RAM / GPU per tenant |
| **Scheduler fair-share** | Volcano / Kueue / **Run:ai** | who runs next when contention arises |

### Per-tenant rate limit (Redis sliding-window, M37 §6)

```python
def allow(tenant_id: str, route: str, limit_per_min: int) -> bool:
    key = f"rl:{tenant_id}:{route}:{int(time.time() // 60)}"
    n = r.incr(key)
    if n == 1: r.expire(key, 120)
    return n <= limit_per_min
```

Three quota tiers most platforms expose:
- **Free / dev** — small RPS, small monthly $ cap, may run on cheaper model tier (M79).
- **Pro** — larger caps, all routes, **prompt caching enabled**.
- **Enterprise** — custom contract, dedicated capacity, SLA, optional silo.

### Fair-share scheduling
**Volcano** and **Kueue** (k8s) and **Run:ai** (commercial) implement gang + fair-share GPU scheduling. Each tenant's running training jobs share GPU time in proportion to their assigned weight — large noisy training jobs can't starve everyone else.

## 7 · Tenant-aware observability

Pair this with M50 + M51. Every metric, log, and trace must carry `tenant_id`.

### OTel attribute
```python
from opentelemetry import trace
span = trace.get_current_span()
span.set_attribute("tenant.id", tenant_id)
span.set_attribute("tenant.tier", "pro")
span.set_attribute("model.id", "tenant-acme")
```

### Prometheus labels (M50)
```python
LLM_REQUESTS = Counter(
    "llm_requests_total",
    "Number of LLM requests",
    ["tenant_id", "tier", "model", "route", "status"],
)
LLM_REQUESTS.labels(tenant_id=tid, tier="pro", model="acme", route="chat", status="ok").inc()
```

> ⚠ Don't over-label. **`tenant_id` is high-cardinality** — at 100k+ tenants, Prometheus bites. Common compromise: **per-tenant counters only at the tier level**, plus a **token-level metering pipeline** (Langfuse / ClickHouse / Snowflake) for the per-tenant slice.

### Dashboards every platform needs
- **Per-tenant cost** (today, this month, MoM) — comes free from M79's tagging.
- **Per-tenant errors** (rate, top error class, p99 latency).
- **Per-tenant abuse** (high RPS, runaway tokens, prompt-injection hits — M74).
- **Tenant freshness** (last RAG ingestion, last LoRA fine-tune, last billing event).

## 8 · Auth + OIDC + scoped tokens

The trust chain that makes the whole thing safe.

```
   user / service ───►  IdP (Auth0 / Cognito / Okta / Entra / Keycloak)
                              │ issues OIDC JWT
                              ▼
                  ┌─── API gateway (verifies JWT) ───┐
                  │  decode claim "tenant_id"        │
                  │  decode claim "scopes"           │
                  └────────────────┬─────────────────┘
                                   │ propagates via header / metadata
                                   ▼
                         services (re-verify claim, enforce scope)
```

**Three rules** that sound boring but stop incidents:

1. **Don't trust client-supplied `tenant_id` for authorisation.** Derive it from the verified JWT claim at the gateway. The downstream may *use* the propagated value, but only after the gateway has signed off.
2. **Per-key scopes** — separate API keys for `read_only`, `write`, `admin`. A leaked read key can't fine-tune; a leaked admin key triggers immediate alarms.
3. **Short-lived tokens** for inter-service calls. 15-minute JWTs > eternal API keys. Pair with service-mesh mTLS (Istio / Linkerd) where the stakes are high.

Multi-tenant LLMs add a fourth: **per-tenant system prompts and tools are part of the security boundary.** Tenant A's chatbot must not see tenant B's `delete_user` tool (M74 §5). The tool catalogue is filtered by `tenant_id` at the agent's discovery step (M64 / M75).

## 9 · Tenant lifecycle

| Phase | What happens | Gotcha |
|---|---|---|
| **Onboard** | provision namespace, DB schema, vector collection, default LoRA, API keys, billing record | atomic — if anything fails, rollback all |
| **Active** | normal operation, monitored | flag idle tenants for downgrade |
| **Suspend** | rate-limit to 0, hide UI, keep data | needs to be reversible within SLA |
| **Trial-end** | gentle email; route to free tier | not deletion |
| **Decommission** | nudge customer to export | give 30-90 days |
| **Delete (GDPR / DSAR)** | **hard delete** all PII, embeddings, fine-tunes, audit logs (per retention policy) | irrevocable; needs verifiable proof |

### GDPR / DSAR considerations
- **Right to access:** be able to export all of a tenant's data inside the SLA (usually 30 days).
- **Right to erasure:** when a tenant says "delete me," you must actually delete — including from **vector DBs, model fine-tunes (LoRAs), training-set caches, log archives**.
- **The hard part:** training data. A fine-tune that learned from tenant data isn't easily "unlearned" — most teams handle this by **never training general models on tenant data without consent** and by isolating per-tenant LoRAs that can be deleted with the tenant.

### Operations
- **Provision via IaC (M48)** — Terraform module that creates the namespace, DB schema, vector collection, billing record in one apply.
- **Delete via the same module** — `terraform destroy --target=module.tenant_xyz` with safe-guards.
- **Audit:** every lifecycle event lands in immutable storage (S3 Object Lock, write-once log).

## 10 · The reference architecture + anti-patterns

```
                            ┌─────────────────────────────────────┐
                            │  IdP (OIDC) + secrets manager       │
                            └──────────────────┬──────────────────┘
                                               │
                            ┌──────────────────▼──────────────────┐
                            │  API Gateway / LiteLLM router       │
                            │   – auth, rate-limit, tenant tag    │
                            │   – per-tier model routing (M79)    │
                            └────┬───────────┬───────────────┬────┘
                                 │           │               │
                ┌────────────────▼─┐    ┌────▼───────┐  ┌────▼─────────┐
                │  Inference pool  │    │  Retriever  │  │  Tools / MCP │
                │  vLLM + LoRA reg │    │  vector DB  │  │  scoped per  │
                │  (M44 + M71)     │    │  (M42 + M69)│  │  tenant (M64)│
                └────────┬─────────┘    └─────┬───────┘  └──────┬───────┘
                         │                    │                  │
                         └──────────┬─────────┴──────────┬───────┘
                                    │                    │
                            ┌───────▼──────┐     ┌───────▼──────┐
                            │  Obs (M50/51)│     │  Cost (M79)  │
                            │  tenant_id   │     │  tenant_id   │
                            │  labels      │     │  labels      │
                            └──────────────┘     └──────────────┘
```

### Anti-patterns
- ❌ **Trusting client `tenant_id`.** Always re-derive from the verified principal.
- ❌ **No fair-share scheduling.** One greedy tenant eats your GPU; everyone else gets 503s.
- ❌ **Cross-tenant vector store with no filter audits.** One missing `tenant_id` filter and you leak.
- ❌ **Same Slack alert for every tenant.** Page volume becomes noise; alert by tier.
- ❌ **One giant shared LoRA**. Defeats the per-tenant point; tenants notice tone bleed.
- ❌ **No tenant in your traces / logs / metrics.** You can't debug or bill.
- ❌ **No way to delete a tenant.** Discovered the day you get a DSAR.
- ❌ **"We'll add multi-tenancy later."** Retrofitting is 10× harder than designing in.

## ✅ Recap
- Four archetypes: B2B SaaS · internal platform · model marketplace · regulated gov. Architecture differs by archetype.
- Four isolation models: **shared · namespaced · silo · dedicated.** Most production B2B SaaS = namespaced default + silo for top customers.
- The **request path** carries `tenant_id`; every persistence and resource layer **enforces** rather than **trusts** it.
- **Data isolation**: tenant filter on every vector DB / DB / feature-store query, integration-tested.
- **Per-tenant LoRA** (M39) on one shared base model (M44) = the cost-effective default for LLM apps.
- **Quotas / rate limits / fair-share scheduling** with Redis + Volcano/Kueue + LiteLLM.
- **Observability** must label every metric / log / trace with `tenant_id` (with cardinality care).
- **OIDC** at the gateway, **short-lived scoped tokens** inside, **mTLS** between services.
- **Tenant lifecycle** must support hard-delete (GDPR / DSAR); design from day one.

Next: **M81 — Capstone — Enterprise AI Platform**.
